# S&P 500 membership + ticker resolution extraction

Pulls point-in-time S&P 500 constituent membership from `crsp.msp500list`
(dated permno-level membership: `start`/`ending` columns) so Table 5 can be
re-run restricted to firms that were actually in the S&P 500 during each
test-period month, rather than the full ~10,587-gvkey universe.

This matters because the full universe includes plenty of small/illiquid
names the GNN's decile sorts can swing on; S&P 500-only is a common
robustness check to see if the result holds among large, liquid, well-covered
firms specifically.

**Cell 1 is run by the user** (interactive WRDS password prompt / `.pgpass`),
same pattern as the other extraction notebooks in this folder. Every cell
after that reuses the `db` connection object it creates.

**Also in this notebook**: resolves our 10,587-gvkey training universe to ticker
symbols (via `wrdsapps_link_crsp_factset.fscrsplink`, which has both `ticker`
and `permno`) and reports how many of those 10,587 firms were ever S&P 500
members during the test window -- our dataset has no ticker column at all
today (only gvkey/permno/ncusip/comnam), so this fills that gap too.

In [1]:
# SQLAlchemy 2.x compatibility patch (identical to the other extraction notebooks) --
# the installed wrds package (3.1.x) predates SQLAlchemy 2.x's stricter
# Connection.execute() API, which requires raw SQL strings to be wrapped in
# sqlalchemy.text(...) rather than passed bare. Guarded so re-running this
# cell doesn't double-wrap and recurse infinitely.
import sqlalchemy as sa
from sqlalchemy.engine import Connection

if not getattr(Connection.execute, "_is_sa2_compat_patch", False):
    _original_execute = Connection.execute

    def _execute_compat(self, statement, *args, **kwargs):
        if isinstance(statement, str):
            statement = sa.text(statement)
        return _original_execute(self, statement, *args, **kwargs)

    _execute_compat._is_sa2_compat_patch = True
    Connection.execute = _execute_compat

import wrds

# Run this cell yourself -- it will prompt for your WRDS username/password
# (or use the .pgpass file already saved from earlier).
db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
Loading library list...
Done


## 1. Pull dated S&P 500 membership (`crsp.msp500list`)

`start`/`ending` give the date range each permno was in the index -- this is
point-in-time correct, unlike a static "current constituents" list.

In [2]:
query = """
    SELECT permno, start, ending
    FROM crsp.msp500list
"""
sp500_membership = db.raw_sql(query, date_cols=['start', 'ending'])
print(sp500_membership.shape)
sp500_membership.head()

(2064, 3)


,permno,start,ending
0,10006,1957-03-01,1984-07-18
1,10030,1957-03-01,1969-01-08
2,10049,1925-12-31,1932-10-01
3,10057,1957-03-01,1992-07-02
4,10078,1992-08-20,2010-01-28


## 2. Expand to a (permno, yyyymm) membership panel, restricted to the OOS window

Only need the test/out-of-sample window used in `table5_replication.py`
(`OOS_START`/`OOS_END`) -- no need to expand the full multi-decade history.

In [3]:
import pandas as pd
import sys
sys.path.insert(0, '.')
import table5_replication as t5

oos_start = pd.to_datetime(str(t5.OOS_START), format='%Y%m')
oos_end = pd.to_datetime(str(t5.OOS_END), format='%Y%m') + pd.offsets.MonthEnd(0)
print('OOS window:', oos_start.date(), '->', oos_end.date())

months = pd.date_range(oos_start, oos_end, freq='MS')

rows = []
for permno, grp in sp500_membership.groupby('permno'):
    for m in months:
        m_end = m + pd.offsets.MonthEnd(0)
        # membership spell overlaps this month if start <= month_end and (ending is null or ending >= month_start)
        in_index = ((grp['start'] <= m_end) & (grp['ending'].isna() | (grp['ending'] >= m))).any()
        if in_index:
            rows.append((permno, int(m.strftime('%Y%m'))))

sp500_panel = pd.DataFrame(rows, columns=['permno', 'yyyymm'])
print(sp500_panel.shape, sp500_panel['permno'].nunique(), 'distinct permnos across', sp500_panel['yyyymm'].nunique(), 'months')

OOS window: 2017-01-01 -> 2024-01-31
(43026, 2) 656 distinct permnos across 85 months


## 3. Map permno -> gvkey via the existing CCM link table

Same crosswalk pattern used throughout this project (`data/ccm_link.parquet`).

In [4]:
ccm = pd.read_parquet('../data/ccm_link.parquet')
ccm = ccm[ccm['linktype'].isin(['LU', 'LC']) & ccm['linkprim'].isin(['P', 'C'])]
ccm_small = ccm[['permno', 'gvkey']].drop_duplicates()
# gvkeys from WRDS are 6-digit zero-padded strings (e.g. '001045'); features.csv
# uses plain-int-string gvkeys (e.g. '1045') -- strip the padding so joins match.
ccm_small['gvkey'] = ccm_small['gvkey'].astype(str).str.lstrip('0')

sp500_gvkey_panel = sp500_panel.merge(ccm_small, on='permno', how='inner').drop_duplicates(subset=['gvkey', 'yyyymm'])
print(sp500_gvkey_panel.shape, sp500_gvkey_panel['gvkey'].nunique(), 'distinct gvkeys')

sp500_gvkey_panel[['gvkey', 'yyyymm']].to_csv('../data/sp500_membership_oos_panel.csv', index=False)
print('Saved: ../data/sp500_membership_oos_panel.csv')

(44191, 3) 665 distinct gvkeys
Saved: ../data/sp500_membership_oos_panel.csv


## 4. Resolve our full 10,587-gvkey universe to ticker symbols

Same crosswalk table used for the GDELT extension (`wrdsapps_link_crsp_factset.fscrsplink`),
this time applied to the *entire* training universe (`data/features.csv`), not
just S&P 500 names.

In [5]:
ticker_query = """
    SELECT DISTINCT ticker, permno, cusip
    FROM wrdsapps_link_crsp_factset.fscrsplink
    WHERE ticker IS NOT NULL
"""
ticker_permno_full = db.raw_sql(ticker_query)
print(ticker_permno_full.shape)

# permno -> gvkey via the same CCM link table
ticker_gvkey = ticker_permno_full.merge(ccm_small, on='permno', how='inner')
ticker_gvkey = ticker_gvkey.dropna(subset=['ticker', 'gvkey']).drop_duplicates(subset=['gvkey'])

# Restrict to our actual training universe
features_gvkeys = pd.read_csv('../data/features.csv', usecols=['gvkey'])['gvkey'].astype(str).unique()
ticker_gvkey['gvkey'] = ticker_gvkey['gvkey'].astype(str)
ticker_gvkey_in_universe = ticker_gvkey[ticker_gvkey['gvkey'].isin(features_gvkeys)]

print(f'{len(ticker_gvkey_in_universe)} of {len(features_gvkeys)} training-universe gvkeys '
      f'resolved to a ticker ({100*len(ticker_gvkey_in_universe)/len(features_gvkeys):.1f}%)')

ticker_gvkey_in_universe[['gvkey', 'ticker', 'permno']].to_csv('../data/gvkey_ticker_crosswalk.csv', index=False)
print('Saved: ../data/gvkey_ticker_crosswalk.csv')

(58583, 3)
3127 of 10587 training-universe gvkeys resolved to a ticker (29.5%)
Saved: ../data/gvkey_ticker_crosswalk.csv


## 5. Overlap: our training universe vs. ever-S&P-500 during the test window

How many of our 10,587 firms were an S&P 500 member in at least one test-period
month, and what fraction of firm-months does that represent.

In [6]:
sp500_gvkeys = set(sp500_gvkey_panel['gvkey'].unique())
universe_gvkeys = set(features_gvkeys)

overlap = sp500_gvkeys & universe_gvkeys
print(f'Training universe: {len(universe_gvkeys)} distinct gvkeys')
print(f'Ever S&P 500 (2017-01 to 2024-01, point-in-time): {len(sp500_gvkeys)} distinct gvkeys')
print(f'Overlap: {len(overlap)} gvkeys ({100*len(overlap)/len(universe_gvkeys):.1f}% of our universe, '
      f'{100*len(overlap)/len(sp500_gvkeys):.1f}% of S&P 500 names found in our universe)')

# with tickers, for a human-readable spot check
overlap_with_tickers = ticker_gvkey_in_universe[ticker_gvkey_in_universe['gvkey'].isin(overlap)]
print()
print('Sample of matched tickers:')
print(overlap_with_tickers.head(20))

Training universe: 10587 distinct gvkeys
Ever S&P 500 (2017-01 to 2024-01, point-in-time): 665 distinct gvkeys
Overlap: 127 gvkeys (1.2% of our universe, 19.1% of S&P 500 names found in our universe)

Sample of matched tickers:
     ticker  permno     cusip   gvkey
165    FSLR   91611  33643310  175404
235    MDLZ   89006  60920710  142953
267     VIA   91063  92553P20  165675
326     FTI   89004  30249U10  142811
740     DPZ   90248  25754A20  160211
1042   AVGO   93002  Y0486S10  180711
1383    CRM   90215  79466L30  157855
1663   TRGP   12476  87612G10  185532
2119    TDC   92293  88076W10  178310
2153    LVS   90505  51783410  161844
2313   FBHS   12981  34964C10  188255
2563    AWK   92614  03042010  179437
2602   SBAC   86996  78410G10  121382
2910   GNRC   93246  36873610  183736
4166   AXON   89031  05464C10  143912
4475   ISRG   88352  46120E60  136725
4821    WBD   22976  93442310  164296
4978     WU   91461  95980210  175263
5022    SAI   91547  78390X10  165123
5026    TPR 